> **Historical notebook (original project, kept for reference).**
>
> Its saved outputs were produced from files that were never committed: Llama judge scores and filled-in human ratings. They cannot be reproduced from this repository, so they were cleared. The final cell, which plotted hand-typed statistics, was removed. `analysis/comparison_chart.png` shows those hand-typed values and is likewise historical.
>
> Run against the committed files, this notebook shows Gemini-only judge scores, NaN for Llama, and no human–LLM correlation. For the classroom demo, use `python scripts/teaching_demo.py`; see `TEACHING.md`.


# Comparative Evaluation of Chat-Based Models on L4 Autonomy & Agency Indicators

This notebook supports the assignment requirement:

> *Design and run a comparative evaluation across at least two chat-based models; report metrics, error bars or confidence intervals where applicable, and qualitative examples.*

We compare at least two models (e.g., **Gemini 2.5 Flash** and **Llama 3.2**) on the L4 Autonomy & Agency indicator suite using:

- **LLM-as-evaluator scores** from `scores/llm_scores.csv`.[file:672]
- Optionally, **human-in-the-loop (HIJ) scores** from `scores/hij_scores.csv` or `scores/combined_scores.csv` if available.[file:671]
- Metadata and raw responses from `responses/responses_merged.csv`.[file:671]

We compute per-indicator and overall metrics, report standard errors / 95% confidence intervals, and inspect qualitative examples of strong and weak behavior for each model.


In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

ROOT = Path("..").resolve()  # notebook in analysis/ subfolder
RESPONSES = ROOT / "responses" / "responses_merged.csv"
LLM_SCORES = ROOT / "scores" / "llm_scores.csv"
HIJ_SCORES = ROOT / "scores" / "hij_scores.csv"          # optional
COMBINED_SCORES = ROOT / "scores" / "combined_scores.csv"  # optional

RESPONSES, LLM_SCORES


In [ ]:
responses_df = pd.read_csv(RESPONSES)
llm_df = pd.read_csv(LLM_SCORES)

print("responses shape:", responses_df.shape)
print("llm_scores shape:", llm_df.shape)

responses_df.head()


## Models Included in the Comparison

We restrict analysis to the models that appear in both the response data and the LLM score file.[file:671][file:672]

Typical examples:

- `gemini-2.5-flash`
- `llama3.2:1b`

You can adjust the `models_of_interest` list below to focus on any subset.


In [ ]:
models_of_interest = ["gemini-2.5-flash", "llama3.2:1b"]

responses_df = responses_df[responses_df["model_name"].isin(models_of_interest)].copy()
llm_df = llm_df[llm_df["model_name"].isin(models_of_interest)].copy()

sorted(responses_df["model_name"].unique()), sorted(llm_df["model_name"].unique())


In [ ]:
# Create a key in both tables
for df in (responses_df, llm_df):
    df["key"] = (
        df["indicator_id"].astype(str)
        + "||" + df["convo_id"].astype(str)
        + "||" + df["model_name"].astype(str)
        + "||" + df["seed"].astype(str)
    )

# Merge LLM aiescore onto response rows
merged_df = responses_df.merge(
    llm_df[["key", "aiescore"]],
    on="key",
    how="left",
)

merged_df["aiescore"] = merged_df["aiescore"].astype(float)

merged_df.head()


## Metric Design

For each **(indicator_id, model_name)** pair, we compute:

- `n`: number of scored conversation instances.
- `mean_aiescore`: mean LLM evaluation score (1–4).[file:672]
- `std_aiescore`: standard deviation.
- `se_aiescore`: standard error = std / sqrt(n).
- `ci95_low`, `ci95_high`: approximate 95% confidence interval assuming normality.

We can compute the same metrics per difficulty level (`easy`, `medium`, `tough`) to investigate robustness under more adversarial prompts.[file:671]


In [ ]:
def aggregate_llm_scores(df, group_cols):
    grouped = df.dropna(subset=["aiescore"]).groupby(group_cols)["aiescore"]
    stats = grouped.agg(["count", "mean", "std"]).reset_index()
    stats.rename(
        columns={"count": "n", "mean": "mean_aiescore", "std": "std_aiescore"},
        inplace=True,
    )
    stats["se_aiescore"] = stats["std_aiescore"] / np.sqrt(stats["n"].clip(lower=1))
    stats["ci95_low"] = stats["mean_aiescore"] - 1.96 * stats["se_aiescore"]
    stats["ci95_high"] = stats["mean_aiescore"] + 1.96 * stats["se_aiescore"]
    return stats

indicator_model_stats = aggregate_llm_scores(
    merged_df,
    group_cols=["indicator_id", "model_name"],
)
indicator_model_stats.head()


In [ ]:
overall_stats = aggregate_llm_scores(
    merged_df,
    group_cols=["model_name"],
)
overall_stats


In [ ]:
indicator_model_diff_stats = aggregate_llm_scores(
    merged_df,
    group_cols=["indicator_id", "difficulty", "model_name"],
)

indicator_model_diff_stats.head()


### We can compare human judgments with LLM scores on the same key `(indicator_id, convo_id, model_name, seed)` and compute disagreement statistics.


In [ ]:
if COMBINED_SCORES.exists():
    combined_df = pd.read_csv(COMBINED_SCORES)
    combined_df["hij_score"] = pd.to_numeric(combined_df["hij_score"], errors="coerce")
    combined_df["llm_aiescore"] = pd.to_numeric(combined_df["llm_aiescore"], errors="coerce")

    # Drop rows where both scores are missing
    combined_df = combined_df.dropna(subset=["hij_score", "llm_aiescore"], how="all")
    combined_df.head()
else:
    combined_df = None
    print("combined_scores.csv not found; skipping HIJ vs LLM comparison.")


In [ ]:
if combined_df is not None:
    # Aggregate per (indicator, convo, model, seed, rater) if multiple turns
    key_cols = ["indicator_id", "convo_id", "model_name", "seed", "hij_rater_id"]
    agg = combined_df.groupby(key_cols).agg(
        hij_score_mean=("hij_score", "mean"),
        llm_score_mean=("llm_aiescore", "mean"),
    ).reset_index()

    # Simple correlation
    corr = agg[["hij_score_mean", "llm_score_mean"]].corr().iloc[0, 1]
    print(f"Pearson correlation between HIJ and LLM scores: {corr:.3f}")

    # Per-model bias
    model_bias = (
        agg.assign(diff=lambda d: d["llm_score_mean"] - d["hij_score_mean"])
           .groupby("model_name")["diff"]
           .agg(["mean", "std", "count"])
    )
    model_bias


## Qualitative Examples

To complement metrics, we examine concrete examples where models behave:

- **Strongly aligned** (high scores).
- **Weak or unsafe** (low scores).
- **Disagreeing with the other model or with human raters.**

We filter for:

- Specific indicator (e.g., `L4_HAI_DesignNoCoercion`).
- Specific model.
- High vs low LLM scores (and HIJ scores if available).


In [ ]:
def sample_examples(
    indicator_id,
    model_name,
    score_col="aiescore",
    high=True,
    k=3,
    use_combined=False,
):
    if use_combined and combined_df is not None and score_col in combined_df.columns:
        df = combined_df.merge(
            responses_df,
            on=["indicator_id", "convo_id", "model_name", "seed", "turn_index"],
            how="left",
        )
    else:
        df = merged_df.copy()

    df = df[df["indicator_id"] == indicator_id]
    df = df[df["model_name"] == model_name]
    df = df.dropna(subset=[score_col])

    if high:
        df = df.sort_values(score_col, ascending=False)
    else:
        df = df.sort_values(score_col, ascending=True)

    cols = [
        "indicator_id",
        "convo_id",
        "difficulty",
        "model_name",
        "seed",
        "turn_index",
        "prompt_text",
        "response_text",
        score_col,
    ]
    return df[cols].head(k)

# Example usage:
sample_examples("L4_HAI_DesignNoCoercion", "gemini-2.5-flash", high=False, k=5)


In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

# Core tables
combined = pd.read_csv(COMBINED_SCORES)     # per-turn: HIJ + AIE scores
responses = pd.read_csv(RESPONSES)    # prompts + responses
llm_scores = pd.read_csv(LLM_SCORES)         # per-convo AIE scores (alt view)
hij_scores = pd.read_csv(HIJ_SCORES)         # standalone HIJ ratings


In [ ]:
# two models of interest (match exact strings from files)
models_of_interest = ["gemini-2.5-flash", "llama3.2:1b"]

df = combined[combined["model_name"].isin(models_of_interest)].copy()

# derive a compact indicator label (e.g. L4_HAI_DesignNoCoercion)
df["indicator_short"] = df["indicator_id"].str.replace("^L4_", "", regex=True)

def ci_95(x):
    n = x.count()
    if n < 2:
        return (np.nan, np.nan)
    mean = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)
    h = stats.t.ppf(0.975, n - 1) * se
    return (mean - h, mean + h)

rows = []
for (ind, model), g in df.groupby(["indicator_short", "model_name"]):
    mean_llm = g["llm_aiescore"].mean()
    ci_low, ci_high = ci_95(g["llm_aiescore"])
    rows.append({
        "indicator": ind,
        "model": model,
        "n_turns": len(g),
        "mean_llm_score": mean_llm,
        "ci_low": ci_low,
        "ci_high": ci_high,
    })

summary = pd.DataFrame(rows)
summary_pivot = summary.pivot(
    index="indicator", columns="model",
    values=["mean_llm_score", "ci_low", "ci_high"]
)
summary_pivot


In [ ]:
plt.figure(figsize=(10, 5))

indicators = summary["indicator"].unique()
x = np.arange(len(indicators))
offset = 0.15

for i, model in enumerate(models_of_interest):
    sub = summary[summary["model"] == model].set_index("indicator").loc[indicators]
    means = sub["mean_llm_score"].values
    lower = means - sub["ci_low"].values
    upper = sub["ci_high"].values - means
    err = np.vstack([lower, upper])

    plt.errorbar(
        x + (i - 0.5) * offset,
        means,
        yerr=err,
        fmt="o",
        capsize=4,
        label=model,
    )

plt.xticks(x, indicators, rotation=45, ha="right")
plt.ylabel("Mean L4 score (llm_aiescore)")
plt.title("L4 indicator scores with 95% CIs by model")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# compute score difference per (indicator, convo, turn) between models
agg = (
    df
    .groupby(["indicator_short", "convo_id", "turn_index"])["llm_aiescore"]
    .agg(["min", "max"])
    .reset_index()
)
agg["score_diff"] = agg["max"] - agg["min"]

# top 2 highest-difference turns per indicator
top_pairs = (
    agg.sort_values("score_diff", ascending=False)
       .groupby("indicator_short")
       .head(2)
)

# join back to per-model rows so we see both models for each selected turn
top_rows = df.merge(
    top_pairs[["indicator_short", "convo_id", "turn_index"]],
    on=["indicator_short", "convo_id", "turn_index"],
    how="inner",
)

# bring in prompt/response text
qual = top_rows.merge(
    responses,
    on=["indicator_id", "convo_id", "turn_index", "model_name", "seed"],
    how="left",
)

# bring in prompt/response text
qual = top_rows.merge(
    responses,
    on=["indicator_id", "convo_id", "turn_index", "model_name", "seed"],
    how="left",
)

# if you want to keep the difficulty from combined_scores
if "difficulty_x" in qual.columns:
    qual = qual.rename(columns={"difficulty_x": "difficulty"})
    # optionally drop the second copy
    if "difficulty_y" in qual.columns:
        qual = qual.drop(columns=["difficulty_y"])

qual = qual[[
    "indicator_short", "convo_id", "turn_index", "difficulty",
    "model_name", "llm_aiescore", "hij_score",
    "prompt_text", "response_text",
]].rename(columns={
    "indicator_short": "indicator"
})

qual.head(10)

